# Wikidata Anthropologist List

Created by [Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

---

## What This Notebook Does

This notebook queries Wikidata for all people with occupation "anthropologist" (Q4773904) and exports a comprehensive list with basic metadata. Use this to get a complete inventory of anthropologists currently in Wikidata — useful for bibliometric analysis, identifying gaps in coverage, or building discipline-specific datasets.

**Note**: This queries the **main Wikidata endpoint** (not the scholarly endpoint), since person items are on the general graph.

## Output Fields

- **QID**: Wikidata identifier
- **Label**: Person's name
- **Description**: Wikidata description
- **Birth Year**: Year of birth (if available)
- **Death Year**: Year of death (if available)
- **Gender**: Gender (if available)
- **Citizenship**: Country of citizenship (if available)
- **ORCID**: ORCID identifier (if available)

## Workflow

1. **Query**: SPARQL query retrieves all anthropologists with optional metadata
2. **Deduplicate**: Consolidates multiple values (e.g., dual citizenship) into single rows
3. **Preview**: Shows data completeness statistics
4. **Export**: Downloads timestamped CSV

## Citation

If you use this notebook, please cite:

> Artz, M. (2026). Wikidata Anthropologist List. GitHub. https://github.com/MattArtzAnthro

*A citable DOI will be available via Zenodo.*

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

## Setup

In [ ]:
!pip install requests pandas -q

import requests
import pandas as pd
from datetime import datetime

print("✓ Setup complete.")

## Configuration

In [ ]:
# Wikidata SPARQL endpoint (general, not scholarly — person items are on the main graph)
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
USER_AGENT = "WikidataAnthropologistList/1.0 (https://www.mattartz.me/)"
TIMEOUT = 120  # seconds

print(f"Endpoint: {WIKIDATA_ENDPOINT}")

## Query All Anthropologists

In [ ]:
SPARQL_QUERY = """
SELECT DISTINCT 
  ?person 
  ?personLabel 
  ?personDescription
  ?birthYear
  ?deathYear
  ?genderLabel
  ?citizenshipLabel
  ?orcid
WHERE {
  ?person wdt:P106 wd:Q4773904 .  # occupation: anthropologist
  ?person wdt:P31 wd:Q5 .          # instance of: human
  
  OPTIONAL { 
    ?person wdt:P569 ?birth .
    BIND(YEAR(?birth) AS ?birthYear)
  }
  OPTIONAL { 
    ?person wdt:P570 ?death .
    BIND(YEAR(?death) AS ?deathYear)
  }
  OPTIONAL { ?person wdt:P21 ?gender . }
  OPTIONAL { ?person wdt:P27 ?citizenship . }
  OPTIONAL { ?person wdt:P496 ?orcid . }
  
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?personLabel
"""

print("Query defined. Running...")

In [ ]:
def run_query():
    """Execute SPARQL query and return deduplicated DataFrame."""
    print("Querying Wikidata for all anthropologists...")

    try:
        response = requests.get(
            WIKIDATA_ENDPOINT,
            params={'query': SPARQL_QUERY, 'format': 'json'},
            headers={'User-Agent': USER_AGENT},
            timeout=TIMEOUT
        )
        response.raise_for_status()
    except requests.exceptions.Timeout:
        print("Query timed out. Try again or increase TIMEOUT.")
        return pd.DataFrame()
    except requests.exceptions.RequestException as e:
        print(f"Query failed: {e}")
        return pd.DataFrame()

    data = response.json()
    bindings = data.get('results', {}).get('bindings', [])

    rows = []
    for b in bindings:
        qid = b.get('person', {}).get('value', '').replace('http://www.wikidata.org/entity/', '')
        rows.append({
            'qid': qid,
            'label': b.get('personLabel', {}).get('value', ''),
            'description': b.get('personDescription', {}).get('value', ''),
            'birth_year': b.get('birthYear', {}).get('value', ''),
            'death_year': b.get('deathYear', {}).get('value', ''),
            'gender': b.get('genderLabel', {}).get('value', ''),
            'citizenship': b.get('citizenshipLabel', {}).get('value', ''),
            'orcid': b.get('orcid', {}).get('value', ''),
            'wikidata_url': f"https://www.wikidata.org/wiki/{qid}"
        })

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    # Deduplicate: OPTIONAL joins can produce multiple rows per person
    # (e.g., dual citizenship, multiple genders). Consolidate with semicolons.
    raw_count = len(df)
    df = df.groupby('qid', as_index=False).agg({
        'label': 'first',
        'description': 'first',
        'birth_year': 'first',
        'death_year': 'first',
        'gender': lambda x: '; '.join(sorted(set(v for v in x if v))),
        'citizenship': lambda x: '; '.join(sorted(set(v for v in x if v))),
        'orcid': 'first',
        'wikidata_url': 'first'
    })

    if raw_count != len(df):
        print(f"   Deduplicated {raw_count:,} rows to {len(df):,} unique people")

    return df

# Run the query
df = run_query()

if not df.empty:
    print(f"\nFound {len(df):,} anthropologists in Wikidata")

## Preview Results

In [ ]:
if df.empty:
    print("No results to display.")
else:
    total = len(df)
    print(f"Total anthropologists: {total:,}\n")

    # Data completeness
    has_birth = df['birth_year'].astype(bool).sum()
    has_death = df['death_year'].astype(bool).sum()
    has_orcid = df['orcid'].astype(bool).sum()
    has_gender = df['gender'].astype(bool).sum()
    has_citizenship = df['citizenship'].astype(bool).sum()

    print("Data Completeness:")
    print(f"   Birth year:  {has_birth:,} ({has_birth/total*100:.1f}%)")
    print(f"   Death year:  {has_death:,} ({has_death/total*100:.1f}%)")
    print(f"   Gender:      {has_gender:,} ({has_gender/total*100:.1f}%)")
    print(f"   Citizenship: {has_citizenship:,} ({has_citizenship/total*100:.1f}%)")
    print(f"   ORCID:       {has_orcid:,} ({has_orcid/total*100:.1f}%)")

    print("\nSample rows:")
    display(df.head(10))

## Export to CSV

In [ ]:
if df.empty:
    print("No data to export.")
else:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"wikidata_anthropologists_{timestamp}.csv"

    df.to_csv(filename, index=False)
    print(f"Saved: {filename} ({len(df):,} rows)")

    # Download (Colab)
    try:
        from google.colab import files
        files.download(filename)
    except ImportError:
        print("(File saved to working directory)")